# data check

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

file_name = '250505_SF_sweep_before'
df = pd.read_csv(f"{file_name}.csv")

ref_col = 'VREF'
value_col = 'VOUT'


def show_section(title):
    display(Markdown(f'### {title}'))


show_section('Dataset')
print(f'file_name : {file_name}.csv')
print(f'rows      : {len(df):,}')
print(f'columns   : {len(df.columns):,}')
print('column names:')
print('  ' + ', '.join(df.columns.astype(str)))

show_section('Preview')
display(df.head())

show_section('Summary Statistics')
display(
    df.describe()
      .T
      .rename_axis('column')
      .style.format('{:.4g}')
)

show_section(f'Grouped {value_col} by {ref_col}')
group_summary = (
    df.groupby(ref_col)[value_col]
      .agg(count='count', min='min', max='max', mean='mean', std='std')
      .reset_index()
)
display(group_summary.style.format({
    ref_col: '{:.4g}',
    'count': '{:,.0f}',
    'min': '{:.4g}',
    'max': '{:.4g}',
    'mean': '{:.4g}',
    'std': '{:.4g}',
}))

max_values = group_summary.set_index(ref_col)['max']
min_values = group_summary.set_index(ref_col)['min']


# Origin GUI


In [ ]:
import html
import pandas as pd
import originpro as op
import ipywidgets as widgets
from IPython.display import display, clear_output

# Clear this cell's old widget output when the cell is re-run.
clear_output(wait=True)

try:
    _origin_gui_box.close()
except Exception:
    pass

file_name_text = widgets.Text(
    value='250505_SF_sweep_before',
    description='file_name',
    layout=widgets.Layout(width='520px')
)

x_text = widgets.Text(value='VIN', description='x', layout=widgets.Layout(width='260px'))
y_text = widgets.Text(value='VOUT', description='y', layout=widgets.Layout(width='260px'))
use_lgd_checkbox = widgets.Checkbox(
    value=True,
    description='Use lgd',
    indent=False,
    layout=widgets.Layout(width='90px')
)
lgd_text = widgets.Text(value='VREF', description='lgd', layout=widgets.Layout(width='260px'))
lgd_box = widgets.HBox([lgd_text, use_lgd_checkbox], layout=widgets.Layout(width='370px'))
title_text = widgets.Text(value='VOUT vs VIN', description='title', layout=widgets.Layout(width='520px'))

load_button = widgets.Button(description='Load columns', button_style='')
run_button = widgets.Button(description='Run Origin', button_style='success')
exit_button = widgets.Button(description='Exit Origin', button_style='warning')
status = widgets.HTML(value='')


def _csv_path():
    name = file_name_text.value.strip()
    if not name.lower().endswith('.csv'):
        name = f'{name}.csv'
    return name


def _read_csv():
    return pd.read_csv(_csv_path())


def _set_status(message):
    status.value = f'<pre style="margin: 8px 0 0 0; white-space: pre-wrap;">{html.escape(str(message))}</pre>'


def _sync_lgd_state(_=None):
    lgd_text.disabled = not use_lgd_checkbox.value


def load_columns(_=None):
    load_button.disabled = True
    try:
        df = _read_csv()
        if x_text.value not in df.columns:
            x_text.value = str(df.columns[0])
        if y_text.value not in df.columns and len(df.columns) > 1:
            y_text.value = str(df.columns[1])
        if use_lgd_checkbox.value and lgd_text.value not in df.columns and len(df.columns) > 2:
            lgd_text.value = str(df.columns[2])
        title_text.value = f'{y_text.value} vs {x_text.value}'
        _set_status('Columns:\n' + ', '.join(df.columns.astype(str)))
    except Exception as exc:
        _set_status(f'Load failed: {exc}')
    finally:
        load_button.disabled = False


def exit_origin(_=None):
    exit_button.disabled = True
    try:
        op.exit()
        _set_status('Origin closed')
    except Exception as exc:
        _set_status(f'Exit failed: {exc}')
    finally:
        exit_button.disabled = False


def _style_layer(gl, x, y, title, show_legend=True):
    gl.rescale()

    gl.lt_exec(f'xb.text$="\\b({x})";')
    gl.lt_exec(f'yl.text$="\\b({y})";')
    gl.lt_exec('xb.font$="Times New Roman";')
    gl.lt_exec('xb.fsize=14;')
    gl.lt_exec('yl.font$="Times New Roman";')
    gl.lt_exec('yl.fsize=14;')

    gl.lt_exec('layer.x.opposite.show=1;')
    gl.lt_exec('layer.y.opposite.show=1;')
    gl.lt_exec('layer.x.thickness=2;')
    gl.lt_exec('layer.y.thickness=2;')
    gl.lt_exec('layer.x.opposite.thickness=2;')
    gl.lt_exec('layer.y.opposite.thickness=2;')

    gl.lt_exec(f'layer.longname$="\\b({title})";')
    gl.lt_exec('layer.title.show=1;')
    gl.lt_exec('legend -r;' if show_legend else 'legend.show=0;')
    gl.lt_exec('layer.width=50;')
    gl.lt_exec('layer.height=60;')
    gl.lt_exec('layer.x.label.fnt.bold=1;')
    gl.lt_exec('layer.y.label.fnt.bold=1;')
    gl.lt_exec('doc -uw;')


def _plot_with_lgd(df, x, y, lgd, title):
    groups = list(df.groupby(lgd))
    combined = {x: sorted(df[x].unique())}
    key_list = []
    for key, g in groups:
        g_sorted = g.sort_values(x)
        combined[f'{lgd}{key}'] = g_sorted[y].values
        key_list.append(key)

    combined_df = pd.DataFrame(combined)
    wks = op.new_sheet('w', lname='AllData')
    wks.from_df(combined_df)
    wks.set_label(0, x, 'L')
    for i, key in enumerate(key_list):
        wks.set_label(i + 1, f'{lgd} = {key}', 'L')

    graph = op.new_graph()
    gl = graph[0]

    color_list = [1, 2, 3, 4, 9]
    symbol_list = [1, 2, 3, 4, 5]

    for i, key in enumerate(key_list):
        dr = op.make_DataRange('X', (wks, 0), 'Y', (wks, i + 1))
        dp = gl.add_plot(dr, type='s')
        if dp is None:
            raise RuntimeError(f'Origin add_plot failed for {lgd}={key}')

        dp.set_int('color', color_list[i % len(color_list)])
        dp.set_int('symbol.kind', symbol_list[i % len(symbol_list)])
        dp.set_int('symbol.interior', 0)
        dp.set_int('symbol.size', 10)

    _style_layer(gl, x, y, title, show_legend=True)
    return len(key_list)


def _plot_without_lgd(df, x, y, title):
    wks = op.new_sheet('w', lname='AllData')
    wks.from_df(df[[x, y]])
    wks.set_label(0, x, 'L')
    wks.set_label(1, y, 'L')

    graph = op.new_graph()
    gl = graph[0]
    dr = op.make_DataRange('X', (wks, 0), 'Y', (wks, 1))
    dp = gl.add_plot(dr, type='s')
    if dp is None:
        raise RuntimeError('Origin add_plot failed for x-y plot')

    dp.set_int('symbol.interior', 0)
    dp.set_int('symbol.size', 10)

    _style_layer(gl, x, y, title, show_legend=False)
    return 1


def run_origin(_=None):
    run_button.disabled = True
    try:
        file_name = file_name_text.value.strip()
        x = x_text.value.strip()
        y = y_text.value.strip()
        use_lgd = use_lgd_checkbox.value
        lgd = lgd_text.value.strip()
        title = title_text.value.strip() or f'{y} vs {x}'

        df = _read_csv()
        required_cols = [x, y] + ([lgd] if use_lgd else [])
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise ValueError(f'Missing column(s): {missing}. Available: {list(df.columns)}')

        try:
            op.exit()
        except Exception:
            pass

        op.set_show(True)

        if use_lgd:
            plot_count = _plot_with_lgd(df, x, y, lgd, title)
            mode_msg = f'lgd={lgd}, groups={plot_count}'
        else:
            plot_count = _plot_without_lgd(df, x, y, title)
            mode_msg = 'lgd off, single x-y plot'

        _set_status(f'Origin graph created: {file_name}\nx={x}, y={y}, {mode_msg}')
    except Exception as exc:
        try:
            op.exit()
        except Exception:
            pass
        _set_status(f'Run failed: {exc}')
        raise
    finally:
        run_button.disabled = False


use_lgd_checkbox.observe(_sync_lgd_state, names='value')
_sync_lgd_state()
load_button.on_click(load_columns)
run_button.on_click(run_origin)
exit_button.on_click(exit_origin)

_origin_gui_box = widgets.VBox([
    widgets.HBox([file_name_text, load_button]),
    widgets.HBox([x_text, y_text, lgd_box]),
    widgets.HBox([title_text, run_button, exit_button]),
    status,
])

display(_origin_gui_box)


# Matplotlib GUI 

In [38]:
import html
import io
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Clear this cell's old widget output when the cell is re-run.
clear_output(wait=True)

try:
    _mpl_gui_box.close()
except Exception:
    pass

try:
    _mpl_gui_generation += 1
except NameError:
    _mpl_gui_generation = 1
_mpl_gui_id = _mpl_gui_generation


def _mpl_is_current(gui_id):
    return gui_id == _mpl_gui_generation


mpl_file_name_text = widgets.Text(
    value='250505_SF_sweep_before',
    description='file_name',
    layout=widgets.Layout(width='520px')
)

mpl_x_text = widgets.Text(value='VIN', description='x', layout=widgets.Layout(width='260px'))
mpl_y_text = widgets.Text(value='VOUT', description='y', layout=widgets.Layout(width='260px'))
mpl_use_lgd_checkbox = widgets.Checkbox(
    value=True,
    description='Use lgd',
    indent=False,
    layout=widgets.Layout(width='90px')
)
mpl_lgd_text = widgets.Text(value='VREF', description='lgd', layout=widgets.Layout(width='260px'))
mpl_lgd_box = widgets.HBox([mpl_lgd_text, mpl_use_lgd_checkbox], layout=widgets.Layout(width='370px'))
mpl_title_text = widgets.Text(value='VOUT vs VIN', description='title', layout=widgets.Layout(width='520px'))

mpl_plot_style = widgets.Dropdown(
    options=[('points', 'points'), ('line + points', 'line+points'), ('line', 'line')],
    value='points',
    description='style',
    layout=widgets.Layout(width='220px')
)
mpl_markerface_dropdown = widgets.Dropdown(
    options=[('none', 'none'), ('filled', 'filled')],
    value='none',
    description='marker',
    layout=widgets.Layout(width='220px')
)
mpl_grid_checkbox = widgets.Checkbox(value=True, description='grid', indent=False, layout=widgets.Layout(width='80px'))

mpl_fig_width = widgets.FloatText(value=8.0, description='fig W', layout=widgets.Layout(width='150px'))
mpl_fig_height = widgets.FloatText(value=5.0, description='fig H', layout=widgets.Layout(width='150px'))
mpl_dpi = widgets.IntText(value=100, description='dpi', layout=widgets.Layout(width='130px'))
mpl_display_width = widgets.IntText(value=900, description='display W', layout=widgets.Layout(width='170px'))
mpl_title_font = widgets.IntText(value=16, description='title fs', layout=widgets.Layout(width='150px'))
mpl_label_font = widgets.IntText(value=12, description='label fs', layout=widgets.Layout(width='150px'))
mpl_tick_font = widgets.IntText(value=10, description='tick fs', layout=widgets.Layout(width='150px'))
mpl_legend_font = widgets.IntText(value=10, description='legend fs', layout=widgets.Layout(width='170px'))
mpl_legend_loc = widgets.Dropdown(
    options=[
        ('outside right', 'outside right'),
        ('upper right', 'upper right'),
        ('upper left', 'upper left'),
        ('lower right', 'lower right'),
        ('lower left', 'lower left'),
        ('best', 'best'),
    ],
    value='outside right',
    description='legend',
    layout=widgets.Layout(width='240px')
)

mpl_load_button = widgets.Button(description='Load columns', button_style='')
mpl_run_button = widgets.Button(description='Run Plot', button_style='success')
mpl_clear_button = widgets.Button(description='Clear Plot', button_style='warning')
mpl_status = widgets.HTML(value='')
mpl_plot_image = widgets.Image(format='png', value=b'', layout=widgets.Layout(width='900px'))


def _mpl_csv_path():
    name = mpl_file_name_text.value.strip()
    if not name.lower().endswith('.csv'):
        name = f'{name}.csv'
    return name


def _mpl_read_csv():
    return pd.read_csv(_mpl_csv_path())


def _mpl_set_status(message):
    mpl_status.value = f'<pre style="margin: 8px 0 0 0; white-space: pre-wrap;">{html.escape(str(message))}</pre>'


def _mpl_sync_lgd_state(_=None):
    mpl_lgd_text.disabled = not mpl_use_lgd_checkbox.value


def mpl_load_columns(_=None, gui_id=_mpl_gui_id):
    if not _mpl_is_current(gui_id):
        return
    mpl_load_button.disabled = True
    try:
        df = _mpl_read_csv()
        if mpl_x_text.value not in df.columns:
            mpl_x_text.value = str(df.columns[0])
        if mpl_y_text.value not in df.columns and len(df.columns) > 1:
            mpl_y_text.value = str(df.columns[1])
        if mpl_use_lgd_checkbox.value and mpl_lgd_text.value not in df.columns and len(df.columns) > 2:
            mpl_lgd_text.value = str(df.columns[2])
        mpl_title_text.value = f'{mpl_y_text.value} vs {mpl_x_text.value}'
        _mpl_set_status('Columns:\n' + ', '.join(df.columns.astype(str)))
    except Exception as exc:
        _mpl_set_status(f'Load failed: {exc}')
    finally:
        mpl_load_button.disabled = False


def mpl_clear_plot(_=None, gui_id=_mpl_gui_id):
    if not _mpl_is_current(gui_id):
        return
    mpl_plot_image.value = b''
    plt.close('all')
    _mpl_set_status('Plot cleared')


def mpl_run_plot(_=None, gui_id=_mpl_gui_id):
    if not _mpl_is_current(gui_id):
        return
    mpl_run_button.disabled = True
    try:
        file_name = mpl_file_name_text.value.strip()
        x = mpl_x_text.value.strip()
        y = mpl_y_text.value.strip()
        use_lgd = mpl_use_lgd_checkbox.value
        lgd = mpl_lgd_text.value.strip()
        title = mpl_title_text.value.strip()
        if not title:
            title = f'{y} vs {x}' if not use_lgd else f'{y} vs {x} for different {lgd}'

        df = _mpl_read_csv()
        required_cols = [x, y] + ([lgd] if use_lgd else [])
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise ValueError(f'Missing column(s): {missing}. Available: {list(df.columns)}')

        markerface = 'none' if mpl_markerface_dropdown.value == 'none' else None
        plot_style = mpl_plot_style.value
        linestyle = 'None' if plot_style == 'points' else '-'
        marker = 'o' if plot_style in ['points', 'line+points'] else None
        fig_width = max(1.0, float(mpl_fig_width.value))
        fig_height = max(1.0, float(mpl_fig_height.value))
        dpi = max(50, int(mpl_dpi.value))
        display_width = max(100, int(mpl_display_width.value))
        title_font = max(1, int(mpl_title_font.value))
        label_font = max(1, int(mpl_label_font.value))
        tick_font = max(1, int(mpl_tick_font.value))
        legend_font = max(1, int(mpl_legend_font.value))

        plt.close('all')
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        try:
            if use_lgd:
                for key, g in df.groupby(lgd):
                    g = g.sort_values(x)
                    ax.plot(
                        g[x],
                        g[y],
                        linestyle=linestyle,
                        marker=marker,
                        markerfacecolor=markerface,
                        markersize=5,
                        label=f'{lgd} = {key}'
                    )
                if mpl_legend_loc.value == 'outside right':
                    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=legend_font)
                else:
                    ax.legend(loc=mpl_legend_loc.value, fontsize=legend_font)
                mode_msg = f'lgd={lgd}, groups={df[lgd].nunique()}'
            else:
                g = df.sort_values(x)
                ax.plot(g[x], g[y], linestyle=linestyle, marker=marker, markerfacecolor=markerface, markersize=5)
                mode_msg = 'lgd off, single x-y plot'

            ax.set_title(title, fontsize=title_font)
            ax.set_xlabel(x, fontsize=label_font)
            ax.set_ylabel(y, fontsize=label_font)
            ax.tick_params(axis='both', labelsize=tick_font)
            ax.grid(mpl_grid_checkbox.value)
            fig.tight_layout()

            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
            mpl_plot_image.layout.width = f'{display_width}px'
            mpl_plot_image.value = buf.getvalue()
        finally:
            plt.close(fig)
            plt.close('all')

        _mpl_set_status(f'Matplotlib plot created: {file_name}\nx={x}, y={y}, {mode_msg}')
    except Exception as exc:
        _mpl_set_status(f'Run failed: {exc}')
        raise
    finally:
        mpl_run_button.disabled = False


mpl_use_lgd_checkbox.observe(_mpl_sync_lgd_state, names='value')
_mpl_sync_lgd_state()
mpl_load_button.on_click(mpl_load_columns)
mpl_run_button.on_click(mpl_run_plot)
mpl_clear_button.on_click(mpl_clear_plot)

_mpl_gui_box = widgets.VBox([
    widgets.HBox([mpl_file_name_text, mpl_load_button]),
    widgets.HBox([mpl_x_text, mpl_y_text, mpl_lgd_box]),
    widgets.HBox([mpl_title_text, mpl_run_button, mpl_clear_button]),
    widgets.HBox([mpl_plot_style, mpl_markerface_dropdown, mpl_grid_checkbox, mpl_legend_loc]),
    widgets.HBox([mpl_fig_width, mpl_fig_height, mpl_dpi, mpl_display_width]),
    widgets.HBox([mpl_title_font, mpl_label_font, mpl_tick_font, mpl_legend_font]),
    mpl_status,
    mpl_plot_image,
])

display(_mpl_gui_box)
